### Calculating the QM solution of the reaction between $OH^-$ and $CH_3Cl$ with five QM water molecules

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyscf import gto, scf
from pyscf.geomopt.geometric_solver import optimize
import pickle

In [2]:
def move_oh_to_distance(coords, target_distance):

    coords = coords.copy()

    O = 0
    H_oh = 1
    C = 2

    carbon = coords[C]
    oxygen = coords[O]

    # Current direction from C toward O
    direction = oxygen - carbon
    direction /= np.linalg.norm(direction)

    # Preserve the O-H vector
    oh_vector = coords[H_oh] - coords[O]

    # Place O at the desired C-O distance
    new_oxygen = carbon + direction * target_distance

    # Move H together with O
    new_hydrogen = new_oxygen + oh_vector

    coords[O] = new_oxygen
    coords[H_oh] = new_hydrogen

    return coords

def build_molecule(coords):

    atom_string = ""

    for symbol, coord in zip(symbols, coords):

        x, y, z = coord

        atom_string += (
            f"{symbol} "
            f"{x:.10f} "
            f"{y:.10f} "
            f"{z:.10f}\n"
        )

    return gto.M(
        atom=atom_string,
        basis="6-31G",
        charge=-1,
        spin=0,
        unit="Angstrom"
    )
    
def write_constraint(distance):

    with open("constraints.txt", "w") as f:

        f.write("$set\n")
        f.write(
            f"distance 1 3 {distance:.8f}\n"
        )

def make_scf(mol):

    mf = scf.RHF(mol)
    return mf
    
def optimize_at_distance(mol, distance, maxsteps=15):

    # Create the constraint file
    write_constraint(distance)

    # Build RHF + implicit solvent
    mf = make_scf(mol)

    # Optimize geometry while keeping C-O fixed
    mol_opt = optimize(
        mf,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # Recalculate final energy using the optimized geometry
    mf_final = make_scf(mol_opt)
    energy = mf_final.kernel()

    # Return the optimized geometry and energy
    return mol_opt, energy, mf_final

def write_xyz(mol, filename, distance):

    coords = mol.atom_coords(
        unit="Angstrom"
    )

    with open(filename, "w") as f:

        f.write(f"{mol.natm}\n")

        f.write(
            f"C-O distance = {distance:.8f} Angstrom\n"
        )

        for symbol, coord in zip(symbols, coords):

            x, y, z = coord

            f.write(
                f"{symbol:2s} "
                f"{x:14.8f} "
                f"{y:14.8f} "
                f"{z:14.8f}\n"
            )


In [3]:
# initial_xyz = """
# O   -5.000   0.000   0.000
# H   -5.970   0.000   0.000

# C    0.000   0.000   0.000
# Cl   1.780   0.000   0.000

# H   -0.630   0.630   0.630
# H   -0.630  -0.630   0.630
# H   -0.630   0.000  -0.890

# O    0.000   3.000   0.000
# H    0.758   3.504   0.000
# H   -0.758   3.504   0.000
# """

In [4]:
initial_xyz = """
O   -5.000   0.000   0.000
H   -5.970   0.000   0.000

C    0.000   0.000   0.000
Cl   1.780   0.000   0.000

H   -0.630   0.630   0.630
H   -0.630  -0.630   0.630
H   -0.630   0.000  -0.890

O    0.000   3.000   0.000
H    0.758   3.504   0.000
H   -0.758   3.504   0.000

O    2.500   1.800   0.000
H    2.900   2.400   0.000
H    1.800   2.100   0.000

O   -1.500   2.000   1.500
H   -0.800   2.300   1.700
H   -1.800   2.700   1.300

O    1.500  -2.000   1.500
H    0.800  -2.300   1.700
H    1.800  -2.700   1.300

O   -1.500  -2.000  -1.500
H   -0.800  -2.300  -1.700
H   -1.800  -2.700  -1.300
"""

In [5]:
# symbols = [
#     "O",
#     "H",
#     "C",
#     "Cl",
#     "H",
#     "H",
#     "H",
#     "O",
#     "H",
#     "H"
# ]

In [6]:
symbols = [
    "O", "H",
    "C", "Cl",
    "H", "H", "H",

    "O", "H", "H",
    "O", "H", "H",
    "O", "H", "H",
    "O", "H", "H",
    "O", "H", "H"
]

In [7]:
mol = gto.M(
    atom=initial_xyz,
    basis="6-31G",
    charge=-1,
    spin=0,
    unit="Angstrom"
)

In [8]:
ndistances = 25
distances = np.linspace(3.0, 1.2, ndistances)

print("Number of steps:", len(distances))
print("Starting distance:", distances[0])
print("Final distance:", distances[-1])

Number of steps: 25
Starting distance: 3.0
Final distance: 1.2


In [9]:
os.makedirs("xyz_frames", exist_ok=True)
for f in glob.glob("xyz_frames/frame_*.xyz"):
    os.remove(f)

print("XYZ folder cleared.")

XYZ folder cleared.


In [10]:
energies = []
actual_distances = []
optimized_molecules = []
mean_field_objects = []

current_mol = mol

In [ ]:
for i, distance in enumerate(distances):

    print("=" * 70)
    print(f"FRAME {i+1}/"+str(ndistances))
    print(f"Target C-O distance: {distance:.6f} Å")

    # ------------------------------------------------
    # 1. Get previous optimized geometry
    # ------------------------------------------------

    current_coords = current_mol.atom_coords(
        unit="Angstrom"
    )

    # ------------------------------------------------
    # 2. Move OH- to the next reaction-coordinate value
    # ------------------------------------------------

    current_coords = move_oh_to_distance(
        current_coords,
        distance
    )

    # ------------------------------------------------
    # 3. Rebuild PySCF molecule
    # ------------------------------------------------

    current_mol = build_molecule(
        current_coords
    )

    # ------------------------------------------------
    # 4. Constrained solvated optimization
    # ------------------------------------------------

    current_mol, energy, mf = optimize_at_distance(
        current_mol,
        distance)
    
    # ------------------------------------------------
    # 5. Store optimized molecule, energy, and mean field
    # ------------------------------------------------

    optimized_molecules.append(current_mol)
    energies.append(energy)
    mean_field_objects.append(mf)
    
    # ------------------------------------------------
    # 6. Calculate actual C-O distance
    # ------------------------------------------------

    optimized_coords = current_mol.atom_coords(
        unit="Angstrom"
    )

    actual_distance = np.linalg.norm(
        optimized_coords[0] -
        optimized_coords[2]
    )

    actual_distances.append(
        actual_distance
    )

    # ------------------------------------------------
    # 7. Save XYZ
    # ------------------------------------------------

    filename = (
        f"xyz_frames/frame_{i:03d}.xyz"
    )

    write_xyz(
        current_mol,
        filename,
        actual_distance
    )

    # ------------------------------------------------
    # 8. Report
    # ------------------------------------------------

    print(
        f"Actual C-O: "
        f"{actual_distance:.6f} Å"
    )

    print(
        f"Energy: "
        f"{energy:.10f} Hartree"
    )

    print(
        f"Saved: {filename}"
    )

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-77ecda09-debf-4727-a630-50fc3a73fa9c.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **

FRAME 1/25
Target C-O distance: 3.000000 Å


66 internal coordinates being used (instead of 66 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-3
Distance 1-2
Distance 3-4
Distance 3-5
Distance 3-6
Distance 3-7
Distance 4-11
Distance 8-9
Distance 8-10
Distance 11-12
Distance 11-13
Distance 14-15
Distance 14-16
Distance 17-18
Distance 17-19
Distance 20-21
Distance 20-22
Angle 4-3-5
Angle 4-3-6
Angle 4-3-7
Angle 5-3-6
Angle 5-3-7
Angle 6-3-7
Angle 3-4-11
Angle 9-8-10
Angle 12-11-13
Angle 15-14-16
Angle 18-17-19
Angle 21-20-22
LinearAngleX 4-11-12
LinearAngleY 4-11-12
Out-of-Plane 11-4-13-12
Dihedral 5-3-4-11
Dihedral 6-3-4-11
Dihedral 7-3-4-11
Dihedral 3-4-11-13
Translation-X 1-2
Translation-X 3-7,11-13
Translation-X 8-10
Translation-X 14-16
Translation-X 17-19
Translation-X 20-22
Translation-Y 1-2
Translation-Y 3-7,11-13
Translation-Y 8-10
Translation-Y 14-16
Translation-Y 17-19
Translation-Y 20-22
Translation-Z 1-2
Translation-Z 3-7,11-13
Translation-Z 8-10
Translation-Z 14-16
Translation-Z 17-19
Transla


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -3.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -3.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000
   O   0.000000   3.000000   0.000000    0.000000  0.000000  0.000000
   H   0.758000   3.504000   0.000000    0.000000  0.000000  0.000000
   H  -0.758000   3.504000   0.000000    0.000000  0.000000  0.000000
   O   2.500000   1.800000   0.000000    0.000000  0.000000  0.000000
   H   2.900000   2.400000   0.000000    0.000000  0.000000  0.000000
   H   1.800000   2.100000   0.0

Step    0 : Gradient = 2.986e-01/6.390e-01 (rms/max) Energy = -953.5108022417
Hessian Eigenvalues: 2.30002e-02 2.33420e-02 4.99921e-02 ... 1.18290e+00 1.34824e+00 1.67645e+00


...


Geometry optimization cycle 7
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -0.893795  -0.408273  -0.348464   -0.003941 -0.000981 -0.005931
   H  -1.484435  -0.876487   0.235812    0.000558  0.002273  0.001763
   C   0.186812  -1.517442  -1.108450    0.003731 -0.001533  0.005706
  Cl   1.795101  -3.388323  -2.767885   -0.004345  0.003922 -0.002347
   H   0.775746  -1.909923  -0.307286    0.009856 -0.009609 -0.003566
   H  -0.446400  -2.229125  -1.591923   -0.001789  0.001429  0.005603
   H   0.716596  -0.872869  -1.773855    0.000985  0.001960  0.005005
   O  -0.786666   2.351666  -0.836690    0.000025  0.000001 -0.000703
   H  -1.223443   2.427353  -1.678908   -0.000040 -0.000901 -0.000615
   H  -0.809588   1.433338  -0.556012    0.000282  0.000276  0.000915
   O   0.835562   3.383986   1.022019   -0.000073 -0.000122 -0.000046
   H   0.189244   3.222378   0.319378   -0.000125 -0.000829  0.000125
   H   1.679362   3.638940   0.6

Step    6 : Displace = 4.378e-03/1.403e-02 (rms/max) Trust = 4.238e-03 (-) Grad_T = 2.364e-04/5.525e-04 (rms/max) E (change) = -954.5289696342 (-9.433e-05) Quality = 1.010
Hessian Eigenvalues: 2.67432e-02 4.12569e-02 4.69598e-02 ... 5.79011e-01 5.80395e-01 5.88330e-01



Geometry optimization cycle 8
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -0.894351  -0.408639  -0.348629   -0.000556 -0.000366 -0.000165
   H  -1.482761  -0.875083   0.239384    0.001673  0.001405  0.003572
   C   0.187971  -1.518676  -1.104984    0.001159 -0.001234  0.003466
  Cl   1.787746  -3.385288  -2.781739   -0.007355  0.003035 -0.013854
   H   0.778791  -1.913498  -0.306011    0.003045 -0.003575  0.001275
   H  -0.447051  -2.228612  -1.588862   -0.000651  0.000513  0.003061
   H   0.716213  -0.871967  -1.769638   -0.000384  0.000902  0.004218
   O  -0.785804   2.350065  -0.837919    0.000862 -0.001601 -0.001229
   H  -1.222814   2.424662  -1.680119    0.000629 -0.002692 -0.001211
   H  -0.809365   1.432137  -0.555966    0.000222 -0.001201  0.000046
   O   0.834372   3.383028   1.021339   -0.001190 -0.000958 -0.000680
   H   0.188954   3.221136   0.317953   -0.000289 -0.001242 -0.001425
   H   1.678129   3.639601   0.6

Step    7 : Displace = 4.370e-03/1.556e-02 (rms/max) Trust = 5.994e-03 (+) Grad_T = 1.127e-04/2.300e-04 (rms/max) E (change) = -954.5289856796 (-1.605e-05) Quality = 1.498
Hessian Eigenvalues: 1.56450e-02 2.84418e-02 4.48643e-02 ... 5.79016e-01 5.80398e-01 5.97786e-01



Geometry optimization cycle 9
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -0.893391  -0.408839  -0.345218    0.000959 -0.000199  0.003411
   H  -1.480299  -0.873447   0.245784    0.002462  0.001636  0.006400
   C   0.187938  -1.519998  -1.101434   -0.000033 -0.001322  0.003550
  Cl   1.773564  -3.378365  -2.808451   -0.014182  0.006923 -0.026712
   H   0.780575  -1.917733  -0.304954    0.001785 -0.004234  0.001057
   H  -0.449273  -2.227920  -1.585502   -0.002223  0.000692  0.003360
   H   0.715901  -0.872608  -1.765778   -0.000311 -0.000641  0.003860
   O  -0.784266   2.346771  -0.841047    0.001537 -0.003294 -0.003128
   H  -1.221365   2.419144  -1.683435    0.001449 -0.005517 -0.003316
   H  -0.808530   1.429688  -0.556041    0.000835 -0.002449 -0.000075
   O   0.832790   3.381325   1.020387   -0.001583 -0.001702 -0.000951
   H   0.188458   3.217372   0.316378   -0.000496 -0.003765 -0.001576
   H   1.675738   3.641909   0.6

### Visualizing some results

In [ ]:
# Looking at the energies
energies_kj=(energies-energies[0])*2626
plt.figure(figsize=(7,5))
plt.plot(actual_distances,energies_kj,marker="o")
plt.gca().invert_xaxis()
xlabel = "C-O distance (Å)"; plt.xlabel(xlabel)
ylabel = "Energy (kJ/mol)"
plt.ylabel(ylabel)
plt.title("Relaxed C-O Reaction Coordinate w/4 solvent waters")
plt.grid(True)
plt.show()

### Saving additional results

In [ ]:
# Saves the optimized molecule objects
with open("optimized_mols.pkl", "wb") as f:
    pickle.dump(optimized_molecules, f)

with open("optimized_mfs.pkl", "wb") as f:
    pickle.dump(mean_field_objects, f)
    
# Save the relative energies as a text file
data = np.array([actual_distances,energies_kj]).T; print(np.shape(data))
np.savetxt('energy.txt',data,header=xlabel+', '+ylabel)

# Save the absolute energies as a text file
data = np.array([actual_distances,energies]).T; print(np.shape(data))
np.savetxt('energy_abs.txt',data,header=xlabel+', '+"Relative energy (Hartree)")